In [1]:

import pandas as pd
from IPython.display import display

# Используем новый "чистый" файл
cash = pd.read_excel(r"C:\Users\PC\Desktop\Приход-Расход.xlsx", sheet_name='Август 2026')
display(cash.head())

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,Дата,Комментарий,Закупка,Доставка,Реклама,Приход,NaN,Результаты,NaN
1,1,NaN,NaN,NaN,NaN,5750,NaN,Приход,364300.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Закупка,87779.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Доставка,0.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Реклама,0.0


In [2]:
colstodrop=['Unnamed: 1','Unnamed: 3','Unnamed: 4','Unnamed: 6','Unnamed: 7','Unnamed: 8']
cash_clean=cash.drop(colstodrop,axis=1)
cash_clean = cash_clean.dropna(how='all')
cash_clean

,Unnamed: 0,Unnamed: 2,Unnamed: 5
0,Дата,Закупка,Приход
1,1,NaN,5750
27,NaN,0,5750
29,Дата,Закупка,Приход
30,2,NaN,1200
...,...,...,...
874,NaN,1457,NaN
875,NaN,NaN,4000
876,NaN,2000,NaN
877,NaN,NaN,2000


In [3]:
# 1. Убираем строки-заголовки (где в первом столбце написано "Дата")
df = cash[~(cash['Unnamed: 0'] == 'Дата')].copy()

# 2. Убираем строки-итоги за день:
#    - в колонке дня (Unnamed: 0) пусто (NaN),
#    - но при этом есть значение в Закупке ИЛИ в Приходе.
# Это и есть те самые «суммы перед следующим блоком».
mask_is_daily_total = (
    df['Unnamed: 0'].isna() & 
    (df['Unnamed: 2'].notna() | df['Unnamed: 5'].notna())
)

df_clean = df[~mask_is_daily_total].copy()

In [4]:
# Переименовываем для удобства
df_clean = df_clean.rename(columns={
    'Unnamed: 0': 'День_raw',
    'Unnamed: 2': 'Закупка_raw',
    'Unnamed: 5': 'Приход_raw'
})

# Преобразуем типы, NaN → 0 для корректного суммирования
df_clean['День'] = pd.to_numeric(df_clean['День_raw'], errors='coerce')
df_clean['Закупка'] = pd.to_numeric(df_clean['Закупка_raw'], errors='coerce').fillna(0)
df_clean['Приход'] = pd.to_numeric(df_clean['Приход_raw'], errors='coerce').fillna(0)

# Считаем общую сумму по всему датасету
total_zakupka = df_clean['Закупка'].sum()
total_prihod = df_clean['Приход'].sum()

print("Всего Закупка:", total_zakupka)
print("Всего Приход:", total_prihod)

Всего Закупка: 16767.0
Всего Приход: 43300.0
